<a href="https://colab.research.google.com/github/brunounder/brunounder/blob/main/Projetos_de_Ci%C3%AAncia_de_Dados_em_Futebol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import sqlite3
import pandas as pd
import plotly.express as px

# 1. CONEXÃO E EXTRAÇÃO (SQL)
# Conectando ao banco de dados SQLite que você subiu
conn = sqlite3.connect('database.sqlite')

# Query profissional unindo Partidas, Ligas e Times
query = """
    SELECT
        L.name AS Liga,
        M.season AS Temporada,
        HT.team_long_name AS Time_Casa,
        VT.team_long_name AS Time_Visitante,
        M.home_team_goal AS Gols_Casa,
        M.away_team_goal AS Gols_Visitante
    FROM Match M
    JOIN League L ON L.id = M.league_id
    JOIN Team HT ON HT.team_api_id = M.home_team_api_id
    JOIN Team VT ON VT.team_api_id = M.away_team_api_id;
"""

df_euro = pd.read_sql_query(query, conn)

# 2. PROCESSAMENTO DE DADOS (Lógica de ADS)
def calcular_resultado(row):
    if row['Gols_Casa'] > row['Gols_Visitante']: return 'Vitória Casa'
    elif row['Gols_Visitante'] > row['Gols_Casa']: return 'Vitória Visitante'
    else: return 'Empate'

df_euro['Resultado'] = df_euro.apply(calcular_resultado, axis=1)
df_euro['Total_Gols'] = df_euro['Gols_Casa'] + df_euro['Gols_Visitante']

# 3. VISUALIZAÇÕES

# --- Gráfico 1: Fator Casa por Liga ---
# Analisa onde é mais difícil ganhar como visitante
fator_casa = df_euro.groupby(['Liga', 'Resultado']).size().reset_index(name='Qtd')
fig1 = px.bar(fator_casa, x='Liga', y='Qtd', color='Resultado',
             title='Equilíbrio de Forças nas Ligas Europeias',
             barmode='group',
             color_discrete_map={'Vitória Casa':'#2ecc71', 'Empate':'#f1c40f', 'Vitória Visitante':'#e74c3c'})
fig1.show()

# --- Gráfico 2: Média de Gols por Liga ---
# Qual campeonato é o mais ofensivo?
media_gols = df_euro.groupby('Liga')['Total_Gols'].mean().reset_index().sort_values('Total_Gols', ascending=False)
fig2 = px.bar(media_gols, x='Total_Gols', y='Liga', orientation='h',
             title='Média de Gols por Partida (Ranking de Ofensividade)',
             color='Total_Gols', color_continuous_scale='Reds')
fig2.show()

# --- Gráfico 3: Distribuição Geral de Resultados ---
fig3 = px.pie(df_euro, names='Resultado', title='Distribuição Geral de Resultados na Europa',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig3.show()

print("Dashboard Europeu finalizado com sucesso!")

Dashboard Europeu finalizado com sucesso!


In [12]:
# 1. Criar colunas de pontos para o Time da Casa e para o Visitante
def calcular_pontos_casa(resultado):
    if resultado == 'Vitória Casa': return 3
    elif resultado == 'Empate': return 1
    else: return 0

def calcular_pontos_visitante(resultado):
    if resultado == 'Vitória Visitante': return 3
    elif resultado == 'Empate': return 1
    else: return 0

df_final['Pts_Casa'] = df_final['Resultado'].apply(calcular_pontos_casa)
df_final['Pts_Visitante'] = df_final['Resultado'].apply(calcular_pontos_visitante)

# 2. Somar pontos por time (agrupando por Time_Casa e Time_Visitante separadamente)
pontos_casa = df_final.groupby('Time_Casa')['Pts_Casa'].sum().reset_index()
pontos_casa.columns = ['Time', 'Pontos']

pontos_visitante = df_final.groupby('Time_Visitante')['Pts_Visitante'].sum().reset_index()
pontos_visitante.columns = ['Time', 'Pontos']

# 3. Unir as duas contagens e somar o total
ranking_geral = pd.concat([pontos_casa, pontos_visitante]).groupby('Time')['Pontos'].sum().reset_index()
ranking_geral = ranking_geral.sort_values(by='Pontos', ascending=False).head(15) # Top 15

# 4. Criar um Gráfico de Barras Horizontal (estilo ranking profissional)
fig_ranking = px.bar(ranking_geral,
                     x='Pontos',
                     y='Time',
                     orientation='h',
                     title='Os 15 Maiores Pontuadores da Europa (2008-2016)',
                     labels={'Pontos': 'Total de Pontos Acumulados', 'Time': 'Clube'},
                     color='Pontos',
                     text='Pontos', # Mostra o valor exato na barra
                     color_continuous_scale='Blues_r')

fig_ranking.update_layout(yaxis={'categoryorder':'total ascending'}) # Do maior para o menor
fig_ranking.show()

In [11]:
# 1. Criar uma coluna com o total de gols da partida
df_final['Total_Gols'] = df_final['Gols_Casa'] + df_final['Gols_Visitante']

# 2. Agrupar por Liga para calcular a média de gols por jogo
media_gols_liga = df_final.groupby('Liga')['Total_Gols'].mean().reset_index()
media_gols_liga = media_gols_liga.sort_values(by='Total_Gols', ascending=False)

# 3. Criar o gráfico de barras verticais
fig_gols = px.bar(media_gols_liga,
                  x='Liga',
                  y='Total_Gols',
                  title='Média de Gols por Partida: Quais as Ligas mais Ofensivas?',
                  labels={'Total_Gols': 'Média de Gols/Jogo', 'Liga': 'Competição'},
                  color='Total_Gols',
                  color_continuous_scale='Reds')

# Ajustar o layout para os nomes das ligas não ficarem cortados
fig_gols.update_layout(xaxis_tickangle=-45)
fig_gols.show()

In [10]:
# 1. Query para buscar atributos dos jogadores e seus nomes
query_jogadores = """
    SELECT
        P.player_name AS Nome,
        PA.overall_rating AS Geral,
        PA.potential AS Potencial,
        PA.finishing AS Finalizacao,
        PA.shot_power AS Forca_Chute
    FROM Player P
    JOIN Player_Attributes PA ON P.player_api_id = PA.player_api_id
"""

df_players_raw = pd.read_sql_query(query_jogadores, conn)

# 2. Como um jogador tem várias linhas (uma por ano), vamos tirar a média por nome
df_players = df_players_raw.groupby('Nome').mean().reset_index()

# 3. Criar o gráfico de dispersão
fig_players = px.scatter(df_players.head(500), # Usando os primeiros 500 para o gráfico carregar rápido
                         x="Finalizacao",
                         y="Potencial",
                         hover_name="Nome",
                         size="Geral",
                         color="Forca_Chute",
                         title="Análise de Scouting: Finalização vs Potencial",
                         labels={'Finalizacao': 'Precisão do Chute', 'Potencial': 'Potencial de Evolução'},
                         color_continuous_scale=px.colors.sequential.Viridis)

fig_players.show()

In [9]:
# 1. Agrupar os dados por Liga e Resultado
analise_liga = df_final.groupby(['Liga', 'Resultado']).size().reset_index(name='Quantidade')

# 2. Criar o gráfico de barras empilhadas
fig_liga = px.bar(analise_liga,
                  x='Liga',
                  y='Quantidade',
                  color='Resultado',
                  title='Equilíbrio de Forças por Liga (Casa vs Visitante)',
                  labels={'Quantidade':'Número de Jogos', 'Liga':'Liga Europeia'},
                  barmode='stack', # Empilha as barras
                  color_discrete_map={'Vitória Casa':'#2ecc71', 'Empate':'#f1c40f', 'Vitória Visitante':'#e74c3c'},
                  height=600)

# Deixar o layout mais limpo e profissional
fig_liga.update_layout(xaxis={'categoryorder':'total descending'}, legend_title_text='Resultado Final')
fig_liga.show()

In [8]:
import plotly.express as px

# 1. Calcular a contagem de resultados (Vitória Casa, Vitória Visitante, Empate)
resumo_resultados = df_final['Resultado'].value_counts().reset_index()
resumo_resultados.columns = ['Resultado', 'Total']

# 2. Calcular a porcentagem para o gráfico
total_jogos = resumo_resultados['Total'].sum()
resumo_resultados['Porcentagem'] = (resumo_resultados['Total'] / total_jogos) * 100

# 3. Criar o gráfico de pizza (Donut Chart) profissional
fig = px.pie(resumo_resultados,
             values='Total',
             names='Resultado',
             title='Distribuição de Resultados: O Fator Casa é Real?',
             hole=0.4, # Transforma em gráfico de rosca (mais moderno)
             color_discrete_map={'Vitória Casa':'#2ecc71', 'Empate':'#f1c40f', 'Vitória Visitante':'#e74c3c'})

# Melhorando o visual
fig.update_traces(textinfo='percent+label', pull=[0.1, 0, 0])
fig.show()

In [7]:
# Query SQL profissional para unir Partidas, Ligas e Times
query_partidas = """
    SELECT
        L.name AS Liga,
        M.season AS Temporada,
        M.stage AS Rodada,
        M.date AS Data,
        HT.team_long_name AS Time_Casa,
        VT.team_long_name AS Time_Visitante,
        M.home_team_goal AS Gols_Casa,
        M.away_team_goal AS Gols_Visitante
    FROM Match M
    JOIN League L ON L.id = M.league_id
    JOIN Team HT ON HT.team_api_id = M.home_team_api_id
    JOIN Team VT ON VT.team_api_id = M.away_team_api_id
    ORDER BY M.date;
"""

# Criando o DataFrame processado
df_final = pd.read_sql_query(query_partidas, conn)

# Criando uma coluna para indicar quem venceu (Lógica para o Dashboard)
def definir_resultado(row):
    if row['Gols_Casa'] > row['Gols_Visitante']:
        return 'Vitória Casa'
    elif row['Gols_Visitante'] > row['Gols_Casa']:
        return 'Vitória Visitante'
    else:
        return 'Empate'

df_final['Resultado'] = df_final.apply(definir_resultado, axis=1)

# Mostrar as primeiras linhas do nosso dataset "limpo"
print("Dados processados com sucesso! Agora temos nomes em vez de números:")
display(df_final.head())

Dados processados com sucesso! Agora temos nomes em vez de números:


,Liga,Temporada,Rodada,Data,Time_Casa,Time_Visitante,Gols_Casa,Gols_Visitante,Resultado
0,Switzerland Super League,2008/2009,1,2008-07-18 00:00:00,BSC Young Boys,FC Basel,1,2,Vitória Visitante
1,Switzerland Super League,2008/2009,1,2008-07-19 00:00:00,FC Aarau,FC Sion,3,1,Vitória Casa
2,Switzerland Super League,2008/2009,1,2008-07-20 00:00:00,FC Luzern,FC Vaduz,1,2,Vitória Visitante
3,Switzerland Super League,2008/2009,1,2008-07-20 00:00:00,Neuchâtel Xamax,FC Zürich,1,2,Vitória Visitante
4,Switzerland Super League,2008/2009,2,2008-07-23 00:00:00,FC Basel,Grasshopper Club Zürich,1,0,Vitória Casa


In [6]:
import sqlite3
import pandas as pd

# 1. Conectar ao arquivo que você subiu no Colab
# Certifique-se que o arquivo aparece na pastinha lateral com este nome exato
conn = sqlite3.connect('database.sqlite')

# 2. Carregar as tabelas principais para a memória (DataFrames)
df_leagues = pd.read_sql_query("SELECT * FROM League", conn)
df_teams   = pd.read_sql_query("SELECT * FROM Team", conn)

# 3. Executar o JOIN (SQL) para ver as Ligas e seus respectivos Países
# Isso é o que chamamos de "Tratamento de Dados" inicial
query_ligas = """
    SELECT L.name AS Liga, C.name AS Pais
    FROM League L
    JOIN Country C ON C.id = L.country_id
"""

df_leagues_names = pd.read_sql_query(query_ligas, conn)

# 4. Mostrar os resultados na tela
print("--- Tabelas carregadas com sucesso! ---")
print("\nLista de Ligas e Países no Dataset:")
print(df_leagues_names)

# Opcional: Mostrar os primeiros 5 times do banco para conferir
print("\nExemplo de alguns times encontrados:")
print(df_teams[['team_long_name']].head())

--- Tabelas carregadas com sucesso! ---

Lista de Ligas e Países no Dataset:
                        Liga         Pais
0     Belgium Jupiler League      Belgium
1     England Premier League      England
2             France Ligue 1       France
3      Germany 1. Bundesliga      Germany
4              Italy Serie A        Italy
5     Netherlands Eredivisie  Netherlands
6         Poland Ekstraklasa       Poland
7   Portugal Liga ZON Sagres     Portugal
8    Scotland Premier League     Scotland
9            Spain LIGA BBVA        Spain
10  Switzerland Super League  Switzerland

Exemplo de alguns times encontrados:
      team_long_name
0           KRC Genk
1       Beerschot AC
2   SV Zulte-Waregem
3   Sporting Lokeren
4  KSV Cercle Brugge
